<div style="background-color:#007033; color:white; padding:18px 22px; border-radius:6px; margin-bottom:10px;"><h1 style="color:white; margin:0 0 4px 0;">Week 4 — Data Pipelines: Batch vs. Streaming</h1><div style="font-size:15px; opacity:0.9;">FraudGuard Case Study · Stage 4 of 10 · INFT 41000, Omar Altrad, Durham College</div></div>

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">This week</b><br>Week 3's diagram named a <b>Feature Pipeline</b> component but didn't build it. This week builds it for real: turn raw transactions into model-ready features, and use a hands-on timing experiment to feel <i>why</i> batch and streaming pipelines are genuinely different engineering problems, not just two settings on the same tool.</div>

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Load Week 3's architecture |
| 2 | Batch vs. streaming: the core trade-off |
| 3 | Orchestration tools (concept, no install) |
| 4 | Engineer features from raw transactions |
| 5 | Batch size experiment |
| 6 | Publish features to S3 |
| 7 | Save your work |

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Setup</b><br>Same shared project as Weeks 1–3, no new packages this week — everything below uses packages already in <code>requirements.txt</code>. First time in this project? Run <code>00_Setup.ipynb</code> first.</div>

In [1]:
import sys
from pathlib import Path

def find_project_root():
    '''Walk up from the current working directory until we find the
    Course_Project root (identified by having both requirements.txt
    and a src/ folder). Works no matter which VS Code cwd setting is in
    effect for notebooks (fileDirname or workspaceFolder), because either
    way this notebook lives somewhere inside the project tree.'''
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the Course_Project root (a folder containing both "
        "requirements.txt and src/). Open this notebook from inside the project "
        "folder in VS Code (File > Open Folder > Course_Project), or set PROJECT_ROOT "
        "manually, e.g. PROJECT_ROOT = Path(r'C:\path\to\INFT41000\Course_Project')."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"

for required_dir in (DATA_DIR, MODELS_DIR):
    if not required_dir.is_dir():
        raise FileNotFoundError(
            f"Missing folder: {required_dir}\n"
            "Create it first (see the folder-structure instructions in 00_Setup.ipynb) -- "
            "this notebook does not create folders for you."
        )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/srujanappu/Desktop/INF_41000_Project


<>:18: SyntaxWarning: invalid escape sequence '\p'
<>:18: SyntaxWarning: invalid escape sequence '\p'
/var/folders/n7/t0kt8c913yq5hswldjl00s580000gn/T/ipykernel_3714/2913787404.py:18: SyntaxWarning: invalid escape sequence '\p'
  "manually, e.g. PROJECT_ROOT = Path(r'C:\path\to\INFT41000\Course_Project')."


In [2]:
import json
import time
import pandas as pd
import numpy as np

print("Setup OK")

Setup OK


## 1. Load Week 3's Architecture

In [3]:
ARCH_PATH = DATA_DIR / "architecture.json"

if not ARCH_PATH.exists():
    raise FileNotFoundError(
        f"Missing {ARCH_PATH.name} in {ARCH_PATH.parent}. Run Week 3's notebook first "
        "(Section 7, \"Save Your Work\") -- this notebook does not regenerate it for you."
    )

with open(ARCH_PATH) as f:
    architecture = json.load(f)

feature_pipeline = next(c for c in architecture["components"] if c["name"] == "Feature Pipeline")
print(json.dumps(feature_pipeline, indent=2))

{
  "name": "Feature Pipeline",
  "type": "data_pipeline",
  "aws_service": "AWS Glue / batch job",
  "mode": "batch",
  "inputs": [
    "S3 Raw Bucket"
  ],
  "outputs": [
    "Model Registry",
    "Drift Monitor"
  ]
}


<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Today we build this box</b><br>Week 3 drew <b>S3 Raw Bucket → Feature Pipeline → Model Registry, Drift Monitor</b> as arrows on a diagram. This week writes the actual code behind the middle box — reading raw data, engineering features, and publishing the result somewhere downstream components can find it.</div>

## 2. Batch vs. Streaming: The Core Trade-off

Every data pipeline picks one of two processing styles (or, in mature systems, both at once
for different needs):

| Dimension | Batch | Streaming |
|---|---|---|
| Processes | A large group of records at once, on a schedule | One record (or a tiny micro-batch) as it arrives |
| Latency | Minutes to hours before results are ready | Milliseconds to seconds |
| Throughput | High — fixed per-run overhead is spread across many rows | Lower per-record overhead cost, but more total overhead across the same volume |
| Typical FraudGuard use | Nightly feature backfill, retraining data for the model | Scoring a transaction the instant it happens (Week 5's live API) |
| Typical tools | AWS Glue, Apache Airflow, pandas/Spark batch jobs | AWS Kinesis, Apache Kafka, Apache Flink |
| Failure recovery | Simple — rerun the whole batch | Harder — must handle partial streams without losing or double-counting records |

FraudGuard needs **both**: a batch pipeline to keep the model's training features fresh (this
week), and a streaming-style path to score each live transaction in milliseconds (Week 5).
They're not competing choices — they're two different jobs.

## 3. Orchestration Tools (Concept — Not Installed This Week)

Real pipelines don't run by someone clicking "Run All" — a scheduler triggers them, retries
failures, and alerts someone when they break. The tools below aren't installed in this
course (they're heavyweight platform services), but you should recognize the names and know
which job they do:

| Tool | Style | What it's for |
|---|---|---|
| Apache Airflow | Batch orchestration | Define pipelines as Python DAGs; schedule, retry, and monitor them |
| AWS Glue | Managed batch ETL | Serverless Spark-based batch jobs, with a built-in data catalog |
| AWS Step Functions | Workflow orchestration | Chains AWS services (Lambda, Glue, SageMaker) into a visual workflow |
| AWS MWAA | Managed Airflow | Airflow, hosted by AWS — same DAGs, no server to maintain |
| Apache Kafka / AWS Kinesis | Streaming | Durable, ordered message streams for real-time data |

Everything below runs directly in this notebook (pandas, no orchestrator) — the same logic
an Airflow DAG or Glue job would run, just without the scheduling/retry layer around it.

## 4. Engineer Features From Raw Transactions

We load Week 1's raw dataset — the same file Week 3 uploaded to S3's `raw/` prefix, playing
the role of the **S3 Raw Bucket** component pulling data in.

In [4]:
RAW_PATH = DATA_DIR / "fraudguard_transactions_week1.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Missing {RAW_PATH.name} in {RAW_PATH.parent}. Run Week 1's notebook first "
        "(it saves this file) -- this notebook does not regenerate it for you."
    )

raw_df = pd.read_csv(RAW_PATH)
print(f"Loaded {len(raw_df):,} raw transactions, {raw_df.shape[1]} columns")
raw_df.head(3)

Loaded 200,000 raw transactions, 11 columns


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,3328.67,C000030216,1188.33,0.00,C000044452,20893.15,24221.82,0,0
1,1,CASH_IN,604.17,C000005110,2896.93,2292.76,C000031905,3574.46,4178.62,0,0
2,1,CASH_IN,1148.42,C000008181,25286.75,24138.33,C000034004,3235.73,4384.15,0,0


### Exercise 4.1
Write `build_features(df)`, adding these seven engineered columns and returning the result (do
not modify the input `df` in place — work on a copy):

| New column | Formula |
|---|---|
| `hour_of_day` | `step % 24` (each step is 1 simulated hour) |
| `balance_delta_orig` | `oldbalanceOrg - newbalanceOrig` |
| `balance_delta_dest` | `newbalanceDest - oldbalanceDest` |
| `orig_error_balance` | `balance_delta_orig - amount` (should be ~0 for a clean transaction — a large deviation is a classic anomaly signal) |
| `dest_error_balance` | `balance_delta_dest - amount` |
| `amount_to_oldbalance_ratio` | `amount / (oldbalanceOrg + 1)` (the `+ 1` avoids dividing by zero) |
| `orig_balance_emptied` | `1` if `newbalanceOrig == 0`, else `0` |

In [5]:
def build_features(df):
    out = df.copy()
    out["hour_of_day"] = out["step"] % 24
    out["balance_delta_orig"] = out["oldbalanceOrg"] - out["newbalanceOrig"]
    out["balance_delta_dest"] = out["newbalanceDest"] - out["oldbalanceDest"]
    out["orig_error_balance"] = out["balance_delta_orig"] - out["amount"]
    out["dest_error_balance"] = out["balance_delta_dest"] - out["amount"]
    out["amount_to_oldbalance_ratio"] = out["amount"] / (out["oldbalanceOrg"] + 1)
    out["orig_balance_emptied"] = (out["newbalanceOrig"] == 0).astype(int)
    return out


features_df = build_features(raw_df)
new_cols = [c for c in features_df.columns if c not in raw_df.columns]
print(f"Added {len(new_cols)} features: {new_cols}")
features_df[new_cols].describe()

Added 7 features: ['hour_of_day', 'balance_delta_orig', 'balance_delta_dest', 'orig_error_balance', 'dest_error_balance', 'amount_to_oldbalance_ratio', 'orig_balance_emptied']


,hour_of_day,balance_delta_orig,balance_delta_dest,orig_error_balance,dest_error_balance,amount_to_oldbalance_ratio,orig_balance_emptied
count,200000.000000,200000.000000,199595.000000,2.000000e+05,1.995950e+05,200000.000000,200000.000000
mean,11.482670,1133.985672,1664.489472,-5.307480e+02,1.072171e-05,1.608719,0.229690
std,6.930606,3381.830179,4123.640484,2.981589e+03,5.003670e-03,10.538311,0.420635
min,0.000000,2.510000,2.510000,-2.887431e+05,-1.000000e-02,0.000080,0.000000
25%,5.000000,238.027500,279.015000,-1.000000e-02,-2.273737e-13,0.058859,0.000000
50%,11.000000,531.125000,669.630000,-1.136868e-13,0.000000e+00,0.224016,0.000000
75%,17.000000,1163.755000,1629.100000,1.136868e-13,2.273737e-13,0.869481,0.000000
max,23.000000,276709.570000,289218.580000,2.605083e+05,1.000000e-02,1323.306644,1.000000


In [6]:
print("Mean orig_error_balance by isFraud:")
print(features_df.groupby("isFraud")["orig_error_balance"].mean())
print("\nMean orig_balance_emptied by isFraud:")
print(features_df.groupby("isFraud")["orig_balance_emptied"].mean())

Mean orig_error_balance by isFraud:
isFraud
0    -561.177077
1    4451.850346
Name: orig_error_balance, dtype: float64

Mean orig_balance_emptied by isFraud:
isFraud
0    0.224986
1    1.000000
Name: orig_balance_emptied, dtype: float64


<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Why these two features stand out</b><br><code>orig_balance_emptied</code> is 1.0 for essentially every fraud case here but only ~0.22 for legitimate ones — a strikingly strong signal, and a big part of why the Week 1 baseline model already did reasonably well. Production teams call this kind of feature a <i>leaky-looking but legitimate</i> signal: it's real, it's causal (this is literally how the fraud pattern behaves), but it's also exactly the kind of feature Week 7's fairness and explainability work needs to scrutinize, not just accept because it scores well.</div>

## 5. Batch Size Experiment

"Batch vs. streaming" isn't just a buzzword difference — batch *size* changes real, measurable
behavior. Smaller batches finish faster individually (lower latency per batch) but have more
total per-call overhead across the whole dataset (lower throughput). Large batches are the
opposite. This is the same trade-off that separates a nightly batch job from a real-time
stream — streaming is really just "batch size of 1," pushed to its latency extreme.

### Exercise 4.2
Write `run_pipeline_in_batches(df, batch_size)` that splits `df` into consecutive chunks of
`batch_size` rows, calls `build_features` on each chunk separately, and times each chunk with
`time.perf_counter()`. Return a tuple of `(combined_result_df, n_batches, batch_times)` where
`batch_times` is a list of per-batch elapsed seconds. Use `np.ceil` to compute the batch count
so a final partial batch is still included.

In [7]:
def run_pipeline_in_batches(df, batch_size):
    n_batches = int(np.ceil(len(df) / batch_size))
    results, batch_times = [], []
    for i in range(n_batches):
        chunk = df.iloc[i * batch_size:(i + 1) * batch_size]
        start = time.perf_counter()
        results.append(build_features(chunk))
        batch_times.append(time.perf_counter() - start)
    return pd.concat(results), n_batches, batch_times


print(f"{'batch_size':>10}  {'n_batches':>9}  {'total_time_s':>12}  {'avg_latency_ms':>14}")
for bs in [500, 5_000, 50_000, len(raw_df)]:
    result, n_batches, batch_times = run_pipeline_in_batches(raw_df, bs)
    total_time = sum(batch_times)
    avg_latency_ms = (total_time / n_batches) * 1000
    print(f"{bs:>10,}  {n_batches:>9,}  {total_time:>12.4f}  {avg_latency_ms:>14.4f}")

batch_size  n_batches  total_time_s  avg_latency_ms


       500        400        0.2735          0.6838
     5,000         40        0.0332          0.8302
    50,000          4        0.0063          1.5867
   200,000          1        0.0065          6.5083


*In your own words: as batch size shrinks from 200,000 down to 500, what happens to total processing time, and what happens to how long any single batch takes to finish? Which of those two numbers matters more for Week 5's live scoring API, and why?*

As batch size shrinks from 200,000 to 500, **total** processing time rises sharply, from about 0.007 s to 0.27 s (roughly 40× slower). Each `build_features` call has a fixed overhead (copying the DataFrame and running seven column operations), and with small batches that overhead is paid 400 times instead of once. At the same time, each **individual** batch finishes much faster: about 0.7 ms for 500 rows vs. about 6.5 ms for all 200,000. For Week 5's live scoring API, per-batch latency matters more. A customer's payment is waiting on a single fraud decision, so what counts is how fast *that one* transaction is scored, not how efficiently we could process a whole day's volume. That's the streaming trade-off: accept lower total throughput in exchange for the low latency that real-time decisions need.

## 6. Publish Features to S3

The Feature Pipeline's job isn't done until its output is somewhere the next component (Model
Registry, and eventually the Scoring API) can find it. We publish the engineered features back
to the same S3 bucket Week 3 created, under a **`features/`** prefix — separate from the
`raw/` prefix, the same way a real data lake separates raw and processed zones.

### Exercise 4.3
Save `features_df` locally, then upload it to `s3://<bucket>/features/fraudguard_features.csv`
using the bucket name from Week 3's saved config. List the bucket's contents afterward,
filtered to the `features/` prefix, to confirm the upload landed.

In [8]:
import boto3
from boto3.exceptions import S3UploadFailedError
from botocore.exceptions import BotoCoreError, ClientError

CONFIG_PATH = DATA_DIR / "aws_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CONFIG_PATH.name} in {CONFIG_PATH.parent}. Run Week 3's notebook first "
        "(Section 8, \"Save Your Work\") -- this notebook does not regenerate it for you."
    )

with open(CONFIG_PATH) as f:
    aws_config = json.load(f)

FEATURES_LOCAL_PATH = DATA_DIR / "fraudguard_features.csv"
FEATURES_S3_KEY = "features/fraudguard_features.csv"

# 1. Save features_df to FEATURES_LOCAL_PATH
features_df.to_csv(FEATURES_LOCAL_PATH, index=False)
print(f"Saved locally: {FEATURES_LOCAL_PATH} ({len(features_df):,} rows)")

# 2. Upload it to aws_config["bucket_name"] at FEATURES_S3_KEY
# 3. List the bucket's contents filtered to Prefix="features/" and print each object
bucket = aws_config["bucket_name"]
s3 = boto3.client("s3", region_name=aws_config.get("region", "us-east-1"))
try:
    s3.upload_file(str(FEATURES_LOCAL_PATH), bucket, FEATURES_S3_KEY)
    print(f"Uploaded to s3://{bucket}/{FEATURES_S3_KEY}")

    listing = s3.list_objects_v2(Bucket=bucket, Prefix="features/")
    for obj in listing.get("Contents", []):
        print(f"  {obj['Key']}  {obj['Size']:,} bytes  {obj['LastModified']}")
except (BotoCoreError, ClientError, S3UploadFailedError) as e:
    # AWS credentials not configured yet -- the S3 step is skipped, local file is still saved
    print(f"S3 upload SKIPPED ({type(e).__name__}): {e}")

Saved locally: /Users/srujanappu/Desktop/INF_41000_Project/data/fraudguard_features.csv (200,000 rows)


S3 upload SKIPPED (S3UploadFailedError): Failed to upload /Users/srujanappu/Desktop/INF_41000_Project/data/fraudguard_features.csv to fraudguard-REPLACE-WITH-YOUR-BUCKET/features/fraudguard_features.csv: An error occurred (InvalidAccessKeyId) when calling the CreateMultipartUpload operation: The AWS Access Key Id you provided does not exist in our records.


## 7. Save Your Work — Needed for Week 5+

In [9]:
pipeline_config = {
    "feature_columns": new_cols,
    "n_rows": len(features_df),
    "s3_bucket": aws_config["bucket_name"],
    "s3_features_key": FEATURES_S3_KEY,
    "local_features_path": str(FEATURES_LOCAL_PATH),
}
with open(DATA_DIR / "pipeline_config.json", "w") as f:
    json.dump(pipeline_config, f, indent=2)

print(f"Saved: {(DATA_DIR / 'pipeline_config.json').resolve()}")
print(json.dumps(pipeline_config, indent=2))

Saved: /Users/srujanappu/Desktop/INF_41000_Project/data/pipeline_config.json
{
  "feature_columns": [
    "hour_of_day",
    "balance_delta_orig",
    "balance_delta_dest",
    "orig_error_balance",
    "dest_error_balance",
    "amount_to_oldbalance_ratio",
    "orig_balance_emptied"
  ],
  "n_rows": 200000,
  "s3_bucket": "fraudguard-REPLACE-WITH-YOUR-BUCKET",
  "s3_features_key": "features/fraudguard_features.csv",
  "local_features_path": "/Users/srujanappu/Desktop/INF_41000_Project/data/fraudguard_features.csv"
}
